# Satellite Image Classification Using Deep Learning

This project builds a convolutional neural network to classify satellite images into four categories: cloudy, desert, green area, and water. The goal is to demonstrate how image classification can be used for environmental monitoring and land-cover analysis.

In [ ]:
# Import libraries 

import os 
import pathlib
import numpy as np 
import matplotlib.pyplot as plt 
import tensorflow as tf 

from tensorflow.keras import layers, models
from tensorflow.keras.utils import image_dataset_from_directory

In [ ]:
# Set dataset path 

data_dir = r"C:\Users\alyss\Downloads\Satelite image data"

data_dir = pathlib.Path(data_dir)

print(data_dir)
print(data_dir.exists())

In [ ]:
# Checking folder/class names 

class_folders = [folder.name for folder in data_dir.iterdir() if folder.is_dir()]
class_folders

In [ ]:
# Count images per class 

for class_name in class_folders:
    image_count = len(list((data_dir / class_name).glob("*")))
    print(f"{class_name}: {image_count} images")

The dataset contains four satellite image categories. Checking the image count for each class helps identify whether the dataset is balanced or if some categories contain more examples than others.

In [ ]:
# laoding training and validation datasets 

img_height = 128 
img_width = 128 
batch_size = 32

train_ds = image_dataset_from_directory( 
    data_dir, 
    validation_split = 0.2, 
    subset="training", 
    seed=42, 
    image_size=(img_height, img_width), 
    batch_size=batch_size
)

val_ds = image_dataset_from_directory(
    data_dir, 
    validation_split=0.2, 
    subset="validation", 
    seed=42, 
    image_size=(img_height, img_width), 
    batch_size=batch_size
)

In [ ]:
# Save class names 

class_names = train_ds.class_names 
print(class_names)

In [ ]:
# Visualize sample images 

plt.figure(figsize=(10,10))

for images, labels in train_ds.take(1):
    for i in range(9):
        ax = plt.subplot(3,3,i +1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]])
        plt.axis("off")

In [ ]:
# improve dataset performance 
AUTOTUNE = tf.data.AUTOTUNE 

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

In [ ]:
# build the CNN model 

num_classes = len(class_names)

model = models.Sequential([
    layers.Rescaling(1./255, input_shape=(img_height, img_width, 3)), 

    layers.Conv2D(32, 3, activation='relu'), 
    layers.MaxPooling2D(), 

    layers.Conv2D(64, 3, activation='relu'), 
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu'), 
    layers.MaxPooling2D(), 

    layers.Flatten(), 
    layers.Dense(128, activation='relu'), 
    layers.Dropout(0.3), 
    layers.Dense(num_classes, activation='softmax')
])

In [ ]:
# Compile the Model

model.compile(
    optimizer='adam', 
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
# View model summary 

model.summary()

This convolutional neural network uses three convolutional layers to detect visual patterns in the satellite images. Max pooling reduces image dimensions while preserving important features. The final dense layers classify each image into one of the four land-cover categories.

In [ ]:
# Train the Model

epochs = 10

history = model.fit(
    train_ds, 
    validation_data=val_ds, 
    epochs=epochs
)

In [ ]:
# Plot accuracy and loss 

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8,5))
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.savefig("figures/accuracy_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.savefig("figures/loss_plot.png", dpi=300, bbox_inches="tight")
plt.show()

The accuracy and loss curves show how well the CNN learned from the training images and how well it generalized to unseen validation images. A large gap between training and validation accuracy may suggest overfitting.

In [ ]:
# Evaluate the model 

val_loss, val_accuracy = model.evaluate(val_ds)

print(f"Validation Loss: {val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
# Confusion matrix 

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

y_true = []
y_pred = []

for images, labels in val_ds: 
    predictions = model.predict(images)
    predicted_labels = np.argmax(predictions, axis = 1)

    y_true.extend(labels.numpy())
    y_pred.extend(predicted_labels)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, 
    display_labels= class_names
)

plt.figure(figsize=(8,8))
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix for Satellite Image Classification')
plt.savefig("figures/confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_true, y_pred, target_names=class_names))

The confusion matrix and classification report provide a more detailed view of model performance. Accuracy shows overall performance, while precision, recall, and F1-score show how well the model performs for each individual satellite image class.

In [ ]:
# Saving work 

os.makedirs("figures", exist_ok=True)
os.makedirs("models", exist_ok=True)

model.save("models/satellite_cnn_model.keras")

In [ ]:
print("Figures saved:")
print(os.listdir("figures"))

print("\nModels saved:")
print(os.listdir("models"))

The final model and evaluation visuals were saved so they can be included in the GitHub repository and referenced in the project README. Saving outputs makes the project more reproducible and easier to present as part of a portfolio.

In [ ]:
# Testing the model on an image 

from tensorflow.keras.utils import load_img, img_to_array 

test_image_path = r"C:\Users\alyss\Downloads\Satelite image data\water\SeaLake_697.jpg"

In [ ]:
img = load_img(test_image_path, target_size=(img_height, img_width))
img_array = img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

predictions = model.predict(img_array)
score = predictions[0]

predicted_class = class_names[np.argmax(score)]
confidence = 100 * np.max(score)

print(f"Predicted class: {predicted_class}")
print(f"Confidence: {confidence:.2f}%")

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.title(f"Prediction: {predicted_class} ({confidence:.2f}%)")
plt.axis("off")
plt.show()

This final prediction test shows how the trained model can classify a new satellite image. The predicted label and confidence score make the model output easier to interpret for a non-technical audience.

## Model Improvement and Retraining

The initial convolutional neural network achieved moderate performance, but some predictions showed low confidence and class confusion between visually similar satellite environments. For example, some water images were incorrectly classified as green areas.

To improve generalization and reduce overfitting, data augmentation techniques were introduced. Data augmentation artificially creates image variation through transformations such as flipping, rotation, and zooming. This helps the model learn more robust visual features and improves performance on unseen images.

The model was then retrained using the augmented dataset and additional training epochs.

### Applying Data Augmentation

Data augmentation increases dataset variability without requiring additional images. This is particularly useful in computer vision tasks where models may otherwise memorize training images instead of learning generalized image patterns.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

### Retraining the CNN Model

After introducing data augmentation, the CNN model was retrained for additional epochs to improve classification performance and prediction confidence.

In [ ]:
epochs = 20
model = models.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),

    data_augmentation,

    layers.Rescaling(1./255),

    layers.Conv2D(32, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(128, 3, activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=epochs
)

In [ ]:
# Evaluate the retrained model 

val_loss, val_accuracy = model.evaluate(val_ds)

print(f"Retrained Validation Loss: {val_loss:.4f}")
print(f"Retrained Validation Accuracy: {val_accuracy:.4f}")

In [ ]:
# Recreate accuracy/loss plots 

acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

loss=history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(epochs)

plt.figure(figsize=(8,5))
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.savefig("figures/accuracy_plot_retrained.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.savefig("figures/loss_plot_retrained.png", dpi=300, bbox_inches="tight")
plt.show()

### Retesting the Model on an Individual Image

After retraining the CNN with data augmentation, the same water image was tested again to see whether the model's prediction improved. This comparison helps evaluate whether the model refinement improved real-world prediction behavior, not just overall validation accuracy.

In [ ]:
test_image_path = r"C:\Users\alyss\Downloads\Satelite image data\water\SeaLake_697.jpg"

img = load_img(test_image_path, target_size=(img_height, img_width))
img_array = img_to_array(img)
img_array = tf.expand_dims(img_array, 0)

predictions = model.predict(img_array)
score = predictions[0]

predicted_class = class_names[np.argmax(score)]
confidence = 100 * np.max(score)

print(f"Predicted class: {predicted_class}")
print(f"Confidence: {confidence:.2f}%")

After retraining with data augmentation, the model correctly classified the test image as water. However, the confidence score remained moderate at 57.77%, suggesting that the image may contain visual features that overlap with other categories, such as shoreline, vegetation, or mixed land-water textures. This result shows improvement in classification accuracy while also highlighting the importance of confidence scores when interpreting computer vision model predictions.

### Original vs. Retrained Model Prediction

| Model Version | Predicted Class | Confidence | Correct? |
|---|---:|---:|---:|
| Original CNN | green_area | 50.00% | No |
| Retrained CNN with Augmentation | water | 57.77% | Yes |

In [ ]:
model.save("models/satellite_cnn_augmented_model.keras")